In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

print("Loading all datasets...")

train        = pd.read_csv(r'C:\Users\Priyanshi Mittal\home-credit-default-risk\data\application_test.csv')
test         = pd.read_csv(r'C:\Users\Priyanshi Mittal\home-credit-default-risk\data\application_test.csv')
bureau       = pd.read_csv(r'C:\Users\Priyanshi Mittal\home-credit-default-risk\data\bureau.csv')
bur_bal      = pd.read_csv(r'C:\Users\Priyanshi Mittal\home-credit-default-risk\data\bureau_balance.csv')
prev_app     = pd.read_csv(r'C:\Users\Priyanshi Mittal\home-credit-default-risk\data\previous_application.csv')
pos_cash     = pd.read_csv(r'C:\Users\Priyanshi Mittal\home-credit-default-risk\data\POS_CASH_balance.csv')
credit_card  = pd.read_csv(r'C:\Users\Priyanshi Mittal\home-credit-default-risk\data\credit_card_balance.csv')
installments = pd.read_csv(r'C:\Users\Priyanshi Mittal\home-credit-default-risk\data\installments_payments.csv')

print(f"train:        {train.shape}")
print(f"test:         {test.shape}")
print(f"bureau:       {bureau.shape}")
print(f"bur_balance:  {bur_bal.shape}")
print(f"prev_app:     {prev_app.shape}")
print(f"pos_cash:     {pos_cash.shape}")
print(f"credit_card:  {credit_card.shape}")
print(f"installments: {installments.shape}")

Loading all datasets...
train:        (48744, 121)
test:         (48744, 121)
bureau:       (1716428, 17)
bur_balance:  (27299925, 3)
prev_app:     (1670214, 37)
pos_cash:     (10001358, 8)
credit_card:  (3840312, 23)
installments: (13605401, 8)


In [3]:
def aggregate_table(df, group_col, prefix):
    """
    Aggregates a supplementary table by group_col.
    - Numerical columns: mean, max, min, sum, std
    - Result is prefixed so we know which table it came from
    """
    num_cols = df.select_dtypes(['int64','float64']).columns.tolist()
    num_cols = [c for c in num_cols if c != group_col]

    agg_dict = {col: ['mean','max','min','sum','std'] for col in num_cols}
    agg_df   = df.groupby(group_col).agg(agg_dict)


    # Flatten multi-level column names
    agg_df.columns = [f"{prefix}_{col}_{stat}" for col, stat in agg_df.columns]
    agg_df = agg_df.reset_index()

    print(f"  {prefix}: {agg_df.shape[1]-1} new features created")
    return agg_df

In [4]:
def engineer_application(df):
    df = df.copy()
    df['DAYS_EMPLOYED_ANOM']  = (df['DAYS_EMPLOYED'] == 365243).astype(int)
    df['DAYS_EMPLOYED']       = df['DAYS_EMPLOYED'].replace(365243, np.nan) 

    df['DAYS_BIRTH']            = df['DAYS_BIRTH'].abs()
    df['DAYS_EMPLOYED']         = df['DAYS_EMPLOYED'].abs()
    df['DAYS_REGISTRATION']     = df['DAYS_REGISTRATION'].abs()
    df['DAYS_ID_PUBLISH']       = df['DAYS_ID_PUBLISH'].abs()
    df['DAYS_LAST_PHONE_CHANGE'] = df['DAYS_LAST_PHONE_CHANGE'].abs()

    df['AGE_YEARS']             = df['DAYS_BIRTH'] / 365
    df['YEARS_EMPLOYED']        = df['DAYS_EMPLOYED'] / 365
    df['YEARS_REGISTRATION']    = df['DAYS_REGISTRATION'] / 365
    df['YEARS_ID_PUBLISH']      = df['DAYS_ID_PUBLISH'] / 365
    df['YEARS_LAST_PHONE_CHANGE'] = df['DAYS_LAST_PHONE_CHANGE'] / 365

    df['AGE_GROUP'] = pd.cut(df['AGE_YEARS'],
                              bins=[0, 25, 35, 45, 55, 100],
                              labels=['<25','25-35','35-45','45-55','55+'])
    
    # Income per family member
    df['INCOME_PER_PERSON']     = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS'].replace(0, 1)

    # Credit-to-income ratio (how much credit relative to income)
    df['CREDIT_INCOME_RATIO']   = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']

    # Annuity-to-income ratio (monthly burden relative to income)
    df['ANNUITY_INCOME_RATIO']  = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']

    # Credit-to-goods ratio (how much is financed vs actual goods price)
    df['CREDIT_GOODS_RATIO']    = df['AMT_CREDIT'] / df['AMT_GOODS_PRICE'].replace(0, np.nan)

    # Down payment proxy (goods price - credit = down payment)
    df['DOWN_PAYMENT']          = df['AMT_GOODS_PRICE'] - df['AMT_CREDIT']
    df['DOWN_PAYMENT_RATIO']    = df['DOWN_PAYMENT'] / df['AMT_GOODS_PRICE'].replace(0, np.nan)

    df['EMPLOYED_TO_AGE_RATIO'] = df['YEARS_EMPLOYED'] / df['AGE_YEARS']

    df['INCOME_PER_EMPLOYED_YEAR'] = df['AMT_INCOME_TOTAL'] / df['YEARS_EMPLOYED'].replace(0, np.nan)

    # Combined score from all 3 external sources
    ext = df[['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']]
    df['EXT_SOURCE_MEAN']       = ext.mean(axis=1)
    df['EXT_SOURCE_MAX']        = ext.max(axis=1)
    df['EXT_SOURCE_MIN']        = ext.min(axis=1)
    df['EXT_SOURCE_STD']        = ext.std(axis=1)
    df['EXT_SOURCE_PROD']       = df['EXT_SOURCE_1'] * df['EXT_SOURCE_2'] * df['EXT_SOURCE_3']

    # Interaction with credit amount
    df['EXT_SOURCE_MEAN_X_CREDIT'] = df['EXT_SOURCE_MEAN'] * df['CREDIT_INCOME_RATIO']

    doc_cols = [c for c in df.columns if 'FLAG_DOCUMENT' in c]
    df['DOCUMENT_COUNT']        = df[doc_cols].sum(axis=1)
    df['HAS_ALL_DOCS']          = (df['DOCUMENT_COUNT'] == len(doc_cols)).astype(int)

    contact_cols = ['FLAG_MOBIL','FLAG_EMP_PHONE','FLAG_WORK_PHONE',
                    'FLAG_CONT_MOBILE','FLAG_PHONE','FLAG_EMAIL']
    df['CONTACT_COUNT']         = df[contact_cols].sum(axis=1)


    df['REGION_MISMATCH_COUNT'] = (
        df['REG_REGION_NOT_LIVE_REGION'].fillna(0) +
        df['REG_REGION_NOT_WORK_REGION'].fillna(0) +
        df['REG_CITY_NOT_LIVE_CITY'].fillna(0) +
        df['REG_CITY_NOT_WORK_CITY'].fillna(0)
    )

    housing_cols = ['FONDKAPREMONT_MODE','HOUSETYPE_MODE',
                    'WALLSMATERIAL_MODE','EMERGENCYSTATE_MODE']
    for col in housing_cols:
        df[f'{col}_MISSING'] = df[col].isnull().astype(int)

    df['LOAN_REPAYMENT_YEARS']  = df['AMT_CREDIT'] / (df['AMT_ANNUITY'].replace(0, np.nan) * 12)

    print(f"Application features: {train.shape[1]} → {df.shape[1]} columns")
    return df
    
    test = engineer_application(test)
    train = engineer_application(train)


In [5]:
bureau.head()

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.000,-153.000,NaN,0,91323.000,0.000,NaN,0.000,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.000,NaN,NaN,0,225000.000,171342.000,NaN,0.000,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.000,NaN,NaN,0,464323.500,NaN,NaN,0.000,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.000,NaN,NaN,0.000,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.000,NaN,77674.500,0,2700000.000,NaN,NaN,0.000,Consumer credit,-21,NaN


In [ ]:
print("Engineering bureau features...")

# ── Step 1: Aggregate bureau_balance INTO bureau ─────────────────
bur_bal_agg = bur_bal.groupby('SK_ID_BUREAU').agg(
    BB_MONTHS_COUNT   = ('MONTHS_BALANCE', 'count'),
    BB_MONTHS_MIN     = ('MONTHS_BALANCE', 'min'),
    BB_STATUS_MEAN    = ('STATUS', lambda x: x.map(
                            {'C':0,'0':1,'1':2,'2':3,'3':4,'4':5,'5':6,'X':np.nan}
                         ).mean())
).reset_index()

# Merge bureau_balance aggregates into bureau
bureau_full = bureau.merge(bur_bal_agg, on='SK_ID_BUREAU', how='left')

# ── Step 2: Compute bureau-level features before final aggregation ─
bureau_full['BUREAU_CREDIT_DEBT_RATIO'] = (
    bureau_full['AMT_CREDIT_SUM_DEBT'] /
    bureau_full['AMT_CREDIT_SUM'].replace(0, np.nan)
)
bureau_full['BUREAU_OVERDUE_RATIO'] = (
    bureau_full['AMT_CREDIT_SUM_OVERDUE'] /
    bureau_full['AMT_CREDIT_SUM'].replace(0, np.nan)
)
bureau_full['BUREAU_IS_ACTIVE']   = (bureau_full['CREDIT_ACTIVE'] == 'Active').astype(int)
bureau_full['BUREAU_IS_CLOSED']   = (bureau_full['CREDIT_ACTIVE'] == 'Closed').astype(int)
bureau_full['BUREAU_IS_BAD_DEBT'] = (bureau_full['CREDIT_ACTIVE'] == 'Bad debt').astype(int)

# ── Step 3: Categorical counts for credit type ──────────────────
credit_type_dummies = pd.get_dummies(bureau_full['CREDIT_TYPE'], prefix='BURO_TYPE')
bureau_full = pd.concat([bureau_full, credit_type_dummies], axis=1)

# ── Step 4: Final aggregation by SK_ID_CURR ─────────────────────
num_cols = bureau_full.select_dtypes(['int64','float64']).columns.tolist()
num_cols = [c for c in num_cols if c not in ['SK_ID_CURR','SK_ID_BUREAU']]

agg_dict = {col: ['mean','max','min','sum','std'] for col in num_cols}

# Extra: count of records per applicant
agg_dict['SK_ID_BUREAU'] = ['count']

bureau_agg = bureau_full.groupby('SK_ID_CURR').agg(agg_dict)
bureau_agg.columns = ['BURO_' + '_'.join(col).upper() 
                       for col in bureau_agg.columns]
bureau_agg = bureau_agg.reset_index()
bureau_agg.rename(columns={'BURO_SK_ID_BUREAU_COUNT':'BURO_TOTAL_CREDITS'}, inplace=True)

print(f"Bureau features created: {bureau_agg.shape[1] - 1}")
print(f"Bureau covers: {bureau_agg['SK_ID_CURR'].nunique():,} unique applicants")

Engineering bureau features...
